# Algorithms for massive datasets project

## 1) Data loading

In [1]:
import os
import zipfile
import sys
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
os.environ['KAGGLE_USERNAME'] = "xxxx"
os.environ['KAGGLE_KEY'] = "xxxx"
!kaggle datasets download -d harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows
with zipfile.ZipFile("imdb-dataset-of-top-1000-movies-and-tv-shows.zip", "r") as zip_ref:
    zip_ref.extractall("imdb_data")

Dataset URL: https://www.kaggle.com/datasets/harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows
License(s): CC0-1.0
imdb-dataset-of-top-1000-movies-and-tv-shows.zip: Skipping, found more recently modified local copy (use --force to force download)


In [2]:
import pandas as pd
from pyspark.sql import SparkSession
import math
from collections import Counter
from itertools import combinations

In [3]:
spark = (
    SparkSession.builder.appName("IMDB MBA").getOrCreate()
    
)
sc = spark.sparkContext

In [4]:
file_path = r"C:\Users\Alberto\Desktop\AMD\imdb_data\imdb_top_1000.csv"
df = spark.read.csv(file_path, header=True, inferSchema=True)
df.head(2)

[Row(Poster_Link='https://m.media-amazon.com/images/M/MV5BMDFkYTc0MGEtZmNhMC00ZDIzLWFmNTEtODM1ZmRlYWMwMWFmXkEyXkFqcGdeQXVyMTMxODk2OTU@._V1_UX67_CR0,0,67,98_AL_.jpg', Series_Title='The Shawshank Redemption', Released_Year='1994', Certificate='A', Runtime='142 min', Genre='Drama', IMDB_Rating=9.3, Overview='Two imprisoned men bond over a number of years, finding solace and eventual redemption through acts of common decency.', Meta_score='80', Director='Frank Darabont', Star1='Tim Robbins', Star2='Morgan Freeman', Star3='Bob Gunton', Star4='William Sadler', No_of_Votes='2343110', Gross='28,341,469'),
 Row(Poster_Link='https://m.media-amazon.com/images/M/MV5BM2MyNjYxNmUtYTAwNi00MTYxLWJmNWYtYzZlODY3ZTk3OTFlXkEyXkFqcGdeQXVyNzkwMjQ5NzM@._V1_UY98_CR1,0,67,98_AL_.jpg', Series_Title='The Godfather', Released_Year='1972', Certificate='A', Runtime='175 min', Genre='Crime, Drama', IMDB_Rating=9.2, Overview="An organized crime dynasty's aging patriarch transfers control of his clandestine empire to 

## 2) Basket creation

In [5]:
actors = ["Star1", "Star2", "Star3", "Star4"]
baskets = (
    df.select(actors).rdd.map(lambda row: [actor.strip()
          for actor in row
          if actor is not None and actor.strip() != ""
      ])
      .filter(lambda basket: len(basket) >= 2).map(lambda basket: sorted(set(basket)))
)

In [6]:
#cash the baskets for SON
baskets = baskets.cache()
n_baskets = baskets.count()

In [7]:
#setting min support
min_support = 0.005
glob_support = math.ceil(min_support * n_baskets)

## 3) Apriori initialization

In [8]:
def loc_freq_items(baskets):
    baskets = [set(basket) for basket in baskets]
    if not baskets:
        return iter([])

    loc_support = math.ceil(min_support * len(baskets))

    item_counts = Counter()
    for basket in baskets:
        for actor in basket:
            item_counts[actor] += 1
    freq = {(actor,) for actor, count in item_counts.items() if count >= loc_support}

    return iter(freq)

## 4) Apriori implementation

In [9]:
#candidate of size k generation

def generate_candidates(freq, k):
    items = sorted({
        item
        for itemset in freq
        for item in itemset
    })

    #combos of size k
    possible_candidates = combinations(items, k)

    candidates = []

    for candidate in possible_candidates:

        subsets = combinations(candidate, k - 1)
        #every subset must be freq
        if all(
            tuple(sorted(subset)) in freq
            for subset in subsets
        ):
            candidates.append(candidate)

    return candidates

In [10]:
def local_apriori(baskets):

    baskets = [set(basket) for basket in baskets]

    if not baskets:
        return iter([])
    
    local_support = math.ceil(
        min_support * len(baskets)
    )

    #freq 1 itemsets
    item_counts = Counter()

    for basket in baskets:
        for actor in basket:
            item_counts[actor] += 1

    #store locally freq itemsets
    all_freq = set(freq)

    k = 2

    while freq:

        candidates = generate_candidates(
            freq,
            k
        )

        if not candidates:
            break

        candidate_counts = Counter()

        for basket in baskets:

            #generate combos of size k
            for candidate in combinations(
                sorted(basket),
                k
            ):

                if candidate in candidates:
                    candidate_counts[candidate] += 1

        #only locally freq candidates
        freq = {
            candidate
            for candidate, count in candidate_counts.items()
            if count >= local_support
        }

        #save
        all_freq.update(freq)

        k += 1

    return iter(all_freq)